# Foundations of Computer Science — Project

Student dropout risk dataset analysis.

In [ ]:
import pandas as pd
import numpy as np

# Load the student data: 100,000 rows (one per student), 34 columns.
# The data file must be in the same folder as this notebook.
df = pd.read_csv('student_dropout_risk_2026.csv.gz')
df.head()

## 1. Students at risk (Online, low attendance, many missed deadlines)

In [ ]:
# Boolean indexing: each condition gives True/False for every row.
# '&' means AND. Each condition needs its own (parentheses), because '&' is
# evaluated before '==', '<' and '>'. The word 'and' does not work on a whole column.
at_risk = df[
    (df['study_mode'] == 'Online') &
    (df['attendance_rate'] < 50) &
    (df['missed_deadlines'] > 5)
]
at_risk  # 197 students

## 2. At-risk students by course type

In [6]:
# groupby splits the at-risk students by course; agg computes 2 numbers per course:
#  - students: how many rows ('count')
#  - dropout_rate: mean of dropout. Dropout is 0/1, so the mean = fraction who dropped out
at_risk_by_course = (
    at_risk.groupby('course_type')
    .agg(students=('person_id', 'count'), dropout_rate=('dropout', 'mean'))
    .sort_values('dropout_rate', ascending=False)  # highest dropout rate first
)
at_risk_by_course

                       students  dropout_rate
course_type                                  
Law & Social Sciences        29      0.896552
Business                     47      0.765957
STEM                         47      0.765957
Arts & Humanities            40      0.750000
Health Sciences              34      0.647059

**Result:** at-risk students drop out a lot (65%–90%). Each group has only 29–47 students, so the differences between courses can be due to chance.

## 3. New columns: pass_rate, gpa_change, gpa_trend

**Fix (floating point):** GPAs have 2 decimals, but subtracting floats can give values such as `-0.5000000000000004` instead of `-0.50`. Then the strict comparison `< -0.5` is wrongly `True`. Rounding `gpa_change` to 2 decimals removes this. Without it, 76 students got the wrong `gpa_trend`.

`units_enrolled` is never 0 in this dataset, so `pass_rate` has no division by zero.

In [9]:
# Vectorized: the operation runs on whole columns at once (no for loop)
df['pass_rate'] = df['units_passed'] / df['units_enrolled']   # units_enrolled is never 0

# GPAs have 2 decimals, but float subtraction can give -0.5000000000000004.
# round(2) makes the comparison with 0.5 / -0.5 safe
df['gpa_change'] = (df['gpa_semester_2'] - df['gpa_semester_1']).round(2)

# np.where(condition, value_if_true, value_if_false); nested to get 3 categories
df['gpa_trend'] = np.where(
    df['gpa_change'] > 0.5, 'improving',
    np.where(df['gpa_change'] < -0.5, 'declining', 'stable')
)
df[['person_id', 'pass_rate', 'gpa_change', 'gpa_trend']].head()

   person_id  pass_rate  gpa_change  gpa_trend
0          1   1.000000       -0.89  declining
1          2   0.785714       -0.03     stable
2          3   0.823529       -0.70  declining
3          4   0.846154       -0.41     stable
4          5   0.875000       -1.05  declining

## 4. Consistency check: units_passed + units_failed == units_enrolled

In [11]:
# Rows where passed + failed is different from enrolled (!=). We expect none.
inconsistent = df[df['units_passed'] + df['units_failed'] != df['units_enrolled']]
print(f"Rows violating the condition: {len(inconsistent)}")  # 0 -> data is consistent
inconsistent

Rows violating the condition: 0


Empty DataFrame
Columns: [person_id, age, gender, nationality, first_generation, disability_status, family_income_level, scholarship_holder, tuition_up_to_date, part_time_job, weekly_work_hours, course_type, study_mode, semester, international_student, admission_grade, previous_qualification_grade, units_enrolled, units_passed, units_failed, gpa_semester_1, gpa_semester_2, attendance_rate, assignments_submitted_rate, library_visits_per_month, lms_logins_per_week, extracurricular_activities, missed_deadlines, stress_level, mental_health_support, satisfaction_score, commute_time_mins, sleep_hours, dropout, pass_rate, gpa_change, gpa_trend]
Index: []

[0 rows x 37 columns]

## 5. Dropout rate by GPA trend, and the 10 students with the largest GPA drop

In [13]:
# Mean of the 0/1 column 'dropout' inside each trend group = dropout rate
dropout_by_trend = df.groupby('gpa_trend')['dropout'].mean()
dropout_by_trend

gpa_trend
declining    0.264795
improving    0.589337
stable       0.391453
Name: dropout, dtype: float64

**Result:** declining students drop out less (26%) than improving ones (59%). This looks strange, but the improving group is small (about 700 students). It shows that GPA change alone is a weak signal.

In [14]:
# Most negative gpa_change = largest drop.
# Sort by gpa_change, then person_id, so ties are broken in a fixed, reproducible way
largest_gpa_drop = (
    df.sort_values(['gpa_change', 'person_id'])
    .head(10)['person_id']
)
largest_gpa_drop

23225    23226
96224    96225
88489    88490
51824    51825
58935    58936
46714    46715
43035    43036
72663    72664
23427    23428
36339    36340
Name: person_id, dtype: int64

**Note (ties):** the 10th place is shared by several students with the same drop. The list above breaks the tie by `person_id`. The table below shows all students tied at the cutoff.

In [16]:
# Check: is the 10th value shared with other students?
cutoff = df['gpa_change'].nsmallest(10).iloc[-1]
print(f"10th largest drop: {cutoff}; students with exactly this drop: {(df['gpa_change'] == cutoff).sum()}")
df.loc[df['gpa_change'] == cutoff, ['person_id', 'gpa_change']].sort_values('person_id')

10th largest drop: -2.23; students with exactly this drop: 4


       person_id  gpa_change
23427      23428       -2.23
36339      36340       -2.23
71899      71900       -2.23
76705      76706       -2.23

## 6. Partition by weekly work hours

In [18]:
# pd.cut puts each value in an interval (a, b]: it excludes a and includes b.
# weekly_work_hours are whole numbers, so: (-0.1,0] = 0, (0,10] = 1-10, (10,20] = 11-20, ...
# -0.1 (not 0) is the lower edge so that the value 0 is included in the first group.
bins = [-0.1, 0, 10, 20, 30, float('inf')]
labels = ['0', '1-10', '11-20', '21-30', 'more than 30']

df['work_hours_group'] = pd.cut(df['weekly_work_hours'], bins=bins, labels=labels)

# sanity check: every student is in exactly one group
assert df['work_hours_group'].notna().all()

df['work_hours_group'].value_counts().reindex(labels)  # reindex = show groups in logical order

work_hours_group
0               51808
1-10             6750
11-20           23923
21-30           15673
more than 30     1846
Name: count, dtype: int64

## 7. Risk score and dropout rate by risk score

In [20]:
# Each condition is True/False; astype(int) turns it into 1/0; then multiply by its points.
# Summing the 6 columns gives one score per student (0 to 8).
risk_score = (
    (df['tuition_up_to_date'] == 0).astype(int) * 2                 # tuition not up to date: 2
    + (df['stress_level'] >= 8).astype(int) * 1                      # stress at least 8: 1
    + (df['sleep_hours'] < 5).astype(int) * 1                        # sleep under 5 h: 1
    + ((df['part_time_job'] == 1) & (df['weekly_work_hours'] > 20)).astype(int) * 1  # job AND >20 h: 1
    + (df['attendance_rate'] < 60).astype(int) * 2                   # attendance under 60: 2
    + (df['missed_deadlines'] > 3).astype(int) * 1                   # more than 3 missed: 1
)
df['risk_score'] = risk_score

# students = how many per score, dropout_rate = fraction who dropped out per score
risk_summary = df.groupby('risk_score').agg(
    students=('person_id', 'count'),
    dropout_rate=('dropout', 'mean')
)
risk_summary

            students  dropout_rate
risk_score                        
0              26955      0.152847
1              24115      0.252084
2              21984      0.337200
3              14557      0.452703
4               7613      0.577433
5               3320      0.673494
6               1149      0.794604
7                267      0.827715
8                 40      0.900000

**Result:** the dropout rate goes up with every point of risk score (15% at score 0, 90% at score 8). So the score works as a rough indicator. Limits: weights are chosen by hand, not learned from data, and the groups with score 7-8 are small.

## 8. GDP per capita dataset (Our World in Data)

In [22]:
gdp_url = (
    "https://ourworldindata.org/grapher/gdp-per-capita-worldbank.csv"
    "?v=1&csvType=full&useColumnShortNames=true"
)
# The site refuses pandas' default request (HTTP error), so we send a browser-like User-Agent
gdp = pd.read_csv(gdp_url, storage_options={"User-Agent": "Mozilla/5.0"})
gdp.head()

        entity code  year  ny_gdp_pcap_pp_kd owid_region
0  Afghanistan  AFG  2000          1617.8264        Asia
1  Afghanistan  AFG  2001          1454.1108        Asia
2  Afghanistan  AFG  2002          1774.3087        Asia
3  Afghanistan  AFG  2003          1815.9282        Asia
4  Afghanistan  AFG  2004          1776.9182        Asia

Rows for aggregates such as `World`, `High-income countries`, or World Bank regional
groupings do not belong to a single continent, so Our World in Data leaves their
`owid_region` empty. Keeping only rows with a non-null `owid_region` removes exactly
those aggregate rows and keeps individual countries.

Then, for each remaining country, keep only the most recent `year`.

In [24]:
# Step 1: keep only rows with a continent -> removes World, income groups, WB regions
gdp_clean = gdp[gdp['owid_region'].notna()]

# Step 2: sort by year, then keep the LAST row of each country = its most recent year
gdp_latest = (
    gdp_clean.sort_values('year')
    .drop_duplicates(subset='entity', keep='last')
)
gdp_latest.head()

                            entity code  year  ny_gdp_pcap_pp_kd  \
6578                         Syria  SYR  2022           4579.440   
7156  United States Virgin Islands  VIR  2022          46479.300   
5778                    San Marino  SMR  2023          70951.790   
2613                     Greenland  GRL  2023          71037.540   
311                          Aruba  ABW  2024          45428.004   

        owid_region  
6578           Asia  
7156  North America  
5778         Europe  
2613  North America  
311   North America  

## 9. Join with student data and compute dropout rate by continent

A few nationalities in the student dataset use short forms (`UK`, `USA`) that differ
from the Our World in Data entity names (`United Kingdom`, `United States`). These are
renamed before merging so the join keys match.

In [27]:
# Students use 'UK' and 'USA'; the GDP file uses 'United Kingdom' and 'United States'.
# Rename in the GDP table so the join keys are identical.
name_fix = {
    'United Kingdom': 'UK',
    'United States': 'USA',
}
gdp_latest = gdp_latest.copy()   # copy: avoids modifying a slice of another dataframe
gdp_latest['entity'] = gdp_latest['entity'].replace(name_fix)

# how='left' keeps ALL students, even if a country has no GDP match (they get NaN)
df_gdp = df.merge(gdp_latest, left_on='nationality', right_on='entity', how='left')
df_gdp[['person_id', 'nationality', 'owid_region', 'ny_gdp_pcap_pp_kd']].head()

   person_id nationality    owid_region  ny_gdp_pcap_pp_kd
0          1      Brazil  South America          20024.697
1          2         USA  North America          76931.310
2          3         USA  North America          76931.310
3          4         USA  North America          76931.310
4          5      Canada  North America          57989.580

Checks before aggregating: the left join must keep one row per student, and `groupby` silently drops rows with a missing `owid_region`, so I count students without a match.

In [29]:
assert len(df_gdp) == len(df)            # no duplicated students after the join
# 0 here = every nationality found a continent, so groupby loses nobody
print('Students without a continent:', df_gdp['owid_region'].isna().sum())
print(df_gdp.loc[df_gdp['owid_region'].isna(), 'nationality'].value_counts())

Students without a continent: 0
Series([], Name: count, dtype: int64)


In [30]:
# Mean of the 0/1 'dropout' column per continent = dropout rate per continent
dropout_by_continent = df_gdp.groupby('owid_region')['dropout'].mean()
dropout_by_continent

owid_region
Africa           0.316721
Asia             0.322908
Europe           0.322762
North America    0.317679
Oceania          0.321981
South America    0.317620
Name: dropout, dtype: float64

In [31]:
# same result with the number of students per continent, to judge how reliable each rate is
df_gdp.groupby('owid_region').agg(students=('person_id', 'count'), dropout_rate=('dropout', 'mean'))

               students  dropout_rate
owid_region                          
Africa             7906      0.316721
Asia              36035      0.322908
Europe            10122      0.322762
North America     38095      0.317679
Oceania             969      0.321981
South America      6873      0.317620

**Result:** dropout is about 32% in every continent (31.7%–32.3%). Continent does not explain dropout here. Oceania has only 969 students, so its rate is the least reliable.

## Use of LLMs / Generative AI

I used Claude AI during this project mainly to help me understand Python code, explain some concepts, and solve coding problems. I also used it to understand pandas functions and to get help when I had errors or did not understand how a part of the code worked.

I did not simply copy the answers. I tried to understand the explanations and tested the code in my Jupyter Notebook to check if it worked correctly. I also checked the results and compared them with the project requirements.

During the project, I made changes to some of the suggested code to make it fit my work and the dataset. I think AI was a useful tool for learning and solving problems, but it was still important for me to understand the code and make my own decisions.